# Laboratorio 9 - Visión Por Computadora

Autores:

- Nelson García
- Joaquín Puente
- Diego Linares

Link al repositorio: https://github.com/Its-Japo/VisionXComputadora/tree/main/Lab_9

# Task 1 - Investigación

### 1. Diferencia fundamental: segmentacion semantica vs segmentacion de instancia

La diferencia exacta entre ambas esta en la etiqueta que recibe cada pixel de la imagen. En segmentacion semantica, el modelo asigna a cada pixel una clase, por ejemplo:

$$S(x,y) \in \{fondo, persona, espada, baculo\}$$

Eso significa que si en la foto aparecen tres personas, todos los pixeles de las tres quedan marcados simplemente como `persona`. El modelo entiende que hay en cada pixel, pero no necesariamente cuantos objetos distintos hay de esa misma clase.

En segmentacion de instancia, la salida ya no es solo una clase, sino una clase asociada a una identidad individual. A nivel conceptual, cada pixel puede entenderse como:

$$I(x,y) = (clase, instancia)$$

Asi, dos personas diferentes dejan de verse como una sola masa de pixeles etiquetada como `persona` y pasan a representarse como `persona_1` y `persona_2`. Lo mismo ocurre con dos espadas identicas: una puede ser `espada_1` y la otra `espada_2`, aunque pertenezcan a la misma categoria visual.

La segmentacion semantica responde la pregunta "de que tipo es este pixel?"; la segmentacion de instancia responde "de que tipo es este pixel y a cual objeto exacto pertenece?". Ese detalle parece pequeno, pero cambia por completo el tipo de decisiones que el sistema puede tomar.

Un ejemplo sencillo en la cabina AR seria este: si hay dos asistentes juntos, la segmentacion semantica puede pintar a ambos como persona y al fondo como fondo, lo cual basta para quitar la convencion y colocar detras la Aldea Oculta de la Hoja. Pero si ambos sostienen una katana parecida, la segmentacion semantica solo dira "aqui hay pixeles de espada"; no podra asegurar cual espada es la del usuario VIP y cual es la del amigo.

### 2. El caso U-Net

U-Net fue propuesto por Ronneberger, Fischer y Brox como una arquitectura de segmentacion basada en una ruta de contraccion (encoder) para capturar contexto y una ruta de expansion (decoder) para recuperar detalle espacial, unidas por skip connections que preservan informacion fina de bordes y contornos [1]. Esa combinacion hace que U-Net sea especialmente fuerte cuando se necesita producir una mascara densa, limpia y precisa a nivel de pixel en una sola pasada.

Para la tarea 1, U-Net encaja muy bien. Si el objetivo es separar automaticamente a las personas del fondo real de la convencion para reemplazarlo por un escenario de anime, el problema puede formularse como una segmentacion relativamente directa: persona contra fondo, o en una variante un poco mas rica, persona, objeto y fondo. En cualquiera de esos casos, U-Net aprovecha dos ventajas muy valiosas.

Primero, su encoder captura suficiente contexto global para distinguir al sujeto principal del entorno, incluso si la escena tiene mucho ruido visual, luces de evento, stands o carteles. Segundo, las conexiones de salto permiten recuperar detalle fino en cabello, hombros, ropa o accesorios, algo importante si se quiere que el recorte final se vea natural y no como una silueta tosca. En otras palabras, para "recortar a los asistentes y cambiarles el fondo", U-Net es una opcion logica, eficiente y bien alineada con el tipo de salida que se necesita.

El problema aparece en la tarea 2. Supongamos que el VIP sostiene una replica de espada y a su lado hay otro asistente con una espada casi igual. Si ambas se cruzan, la salida de U-Net tendera a marcar los pixeles de las dos dentro de la misma categoria espada, porque su cabeza de salida trabaja por clases, no por instancias. Tecnicamente, U-Net produce mapas por clase y luego decide, pixel por pixel, cual clase domina. No existe, dentro de su formulacion semantica estandar, una identidad separada para la espada del usuario y otra para la espada del acompanante.

Por eso, cuando dos objetos de la misma clase se tocan, o tienen formas muy parecidas, U-Net puede unirlos en una sola region, segmentarlos sin identidad consistente o simplemente marcar ambos como el mismo tipo de objeto. Incluso si lograra dibujar dos regiones separadas, todavia faltaria una respuesta crucial: cual de esas dos espadas pertenece al usuario al que quiero aplicarle el brillo digital? Esa ya no es una pregunta puramente semantica; es una pregunta de instancia y de asociacion entre objetos.

En resumen, U-Net es excelente para separar todos los humanos del fondo, pero por si solo no es la arquitectura correcta para seleccionar la espada especifica del usuario cuando hay varias armas parecidas dentro de la misma escena.

### 3. El caso Mask R-CNN

Mask R-CNN fue disenado justamente para resolver el tipo de ambiguedad que U-Net no resuelve bien. Segun He et al., el modelo extiende a Faster R-CNN anadiendo una rama paralela que predice una mascara para cada objeto detectado [2]. La idea clave es que ya no trabaja solo con un mapa global de clases, sino con objetos candidatos individuales.

Su logica de dos etapas viene heredada de Faster R-CNN [3]. En la primera etapa, una Region Proposal Network (RPN) propone regiones donde probablemente hay objetos. En la segunda, cada region se analiza por separado para clasificarla, ajustar su caja delimitadora y producir una mascara binaria propia del objeto. Mask R-CNN ademas introduce RoIAlign, que mejora la alineacion espacial entre la region detectada y la mascara, algo importante cuando el contorno exacto si importa [2].

Ese cambio arquitectonico resuelve el problema de fondo. En vez de generar un unico canal espada para toda la imagen, Mask R-CNN puede generar una mascara para espada_1 y otra para espada_2. En otras palabras, detecta primero que hay dos objetos distintos y solo despues segmenta cada uno con precision de pixel. Gracias a eso, aunque dos espadas identicas esten tocandose o cruzadas, el sistema todavia puede tratarlas como entidades separadas porque cada una tiene su propia propuesta, su propia caja y su propia mascara.

Llevado al contexto de la cabina AR, eso permite algo muy concreto: iluminar la espada del usuario en rojo y la del amigo en azul, incluso si visualmente son casi identicas. Cada arma puede recibir un identificador distinto, y a partir de ese identificador se le puede asignar un efecto visual distinto. Si ademas el sistema sabe cual persona es el VIP, entonces puede vincular la mascara del arma con la persona correcta y aplicar el brillo unicamente sobre esa instancia.

Por eso, desde un punto de vista tecnico, la diferencia no es solo de precision, sino de naturaleza del problema. U-Net responde muy bien a la pregunta "que clase aparece en cada pixel?". Mask R-CNN responde "que objeto especifico esta aqui y cuales pixeles le pertenecen?". Para quitar el fondo de la convencion, U-Net es ideal. Para distinguir la espada correcta entre varias armas parecidas y superponerle un efecto individual, la opcion adecuada es Mask R-CNN.

### Conclusion

Si la mesa directiva tuviera que tomar una decision practica, la recomendacion seria separar las tareas: **U-Net para el reemplazo de fondo** y **Mask R-CNN para identificar e intervenir objetos individuales**, como la espada o el baculo del usuario VIP. De hecho, una cabina AR robusta podria combinar ambas ideas: primero recortar bien a las personas y luego detectar, separar y realzar unicamente la instancia del arma que interesa.

### Referencias principales

[1] Ronneberger, O., Fischer, P. y Brox, T. *U-Net: Convolutional Networks for Biomedical Image Segmentation*. MICCAI 2015 / arXiv. Disponible en: https://arxiv.org/abs/1505.04597

[2] He, K., Gkioxari, G., Dollar, P. y Girshick, R. *Mask R-CNN*. ICCV 2017. Disponible en: https://openaccess.thecvf.com/content_ICCV_2017/html/He_Mask_R-CNN_ICCV_2017_paper.html

[3] Ren, S., He, K., Girshick, R. y Sun, J. *Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks*. NeurIPS 2015. Disponible en: https://papers.nips.cc/paper/5638-faster-r-cnn-towards-real-time-object-detection-with-region-proposal-networks


# Task 2 

1. Explique matemáticamente, utilizando la fórmula del IoU (Intersección sobre Unión), por qué el
algoritmo NMS está causando que los clones superpuestos desaparezcan (Falsos Negativos).

R//

El algoritmo $\textbf{Non-Maximum Suppression}$ utiliza la métrica $\textbf{Intersection over Union (IoU)}$ para eliminar detecciones redundantes. Esta métrica se define como:


$IoU(B,B^*)$ = $\frac{|B \cap B^*|}{|B \cup B^*|}$

donde:

$|B \cup B^*| = |B| + |B^*| - |B \cap B^*|$

El procedimiento de NMS es el siguiente:

- Ordenar las cajas por $\textit{score}$ de confianza.

-  Seleccionar la caja con mayor $\textit{score}$.

- Eliminar todas las cajas cuya superposición con la seleccionada cumpla:

$IoU(B,B^*) > \tau_{NMS}$

- Repetir el proceso con las cajas restantes.

En el caso de múltiples personas muy juntas, abrazadas y parcialmente ocluidas, las cajas correspondientes a individuos distintos pueden presentar un valor alto de IoU. Por ejemplo:


$IoU(B_A,B_B)=0.68$


Si el umbral de NMS es $(\tau_{NMS}=0.5)$, entonces se cumple:


0.68 > 0.5 $\Rightarrow B_B \text{ es eliminada}$

Aunque $(B_B)$ represente realmente a otra persona distinta, NMS la suprime por interpretarla como una detección duplicada. Esto provoca que una detección válida desaparezca del resultado final, generando un $\textbf{Falso Negativo}$.

NMS asume que cajas con alto IoU representan el mismo objeto, lo cual no necesariamente es cierto en escenas densas. Por eso, cuando hay muchas personas superpuestas, varias detecciones correctas desaparecen.


2. Si usted es el ingeniero a cargo y solo puede modificar los hiperparámetros en el código durante la inferencia: ¿Qué pasaría si ajusta el umbral de IoU del NMS a 0.15? ¿Qué pasaría si lo ajusta a 0.95? Justifique qué valor sería más adecuado para este problema de alta densidad.

R//

#### Caso A: ${(\tau_{NMS}=0.15)}$

Si el umbral se reduce a $(0.15)$, NMS se vuelve extremadamente agresivo. En este caso, incluso pequeñas superposiciones entre cajas harán que una de ellas sea eliminada.

Las consecuencias son:

- se suprimen demasiadas cajas,
- disminuye el $\textit{recall}$,
- aumentan los $\textbf{Falsos Negativos}$,
- desaparecen aún más personas en la escena.

#### Caso B: ${(\tau_{NMS}=0.95)}$

Si el umbral se incrementa a $(0.95)$, NMS se vuelve muy permisivo. Solo eliminará cajas casi idénticas.

Las consecuencias son:

- se conservan más detecciones válidas de personas distintas,
- disminuyen los $\textbf{falsos negativos}$ causados por oclusión,
- aumentan los $\textbf{Falsos Positivos}$ por cajas duplicadas.

#### Valor más adecuado

Para escenas de alta densidad, el principal problema es la supresión excesiva de cajas válidas. Por ello, entre $(0.15)$ y $(0.95)$, el valor más adecuado es:

$\tau_{NMS}=0.95$

Este valor permite preservar mejor las detecciones de personas distintas, aunque estén muy próximas o superpuestas. En otras palabras, para este problema es preferible tolerar algunas cajas duplicadas antes que perder instancias reales.


3. Si el presupuesto le permitiera cambiar el modelo a YOLOv10, explique cómo su
arquitectura Asignación Dual de Etiquetas (Dual Label Assignment) resolvería este problema de
oclusión sin necesidad de tocar el NMS.

R//

Los modelos YOLO tradicionales producen múltiples predicciones por objeto durante el entrenamiento, siguiendo una lógica $\textit{one-to-many}$. Esto mejora la supervisión, pero durante inferencia genera redundancia, por lo que se requiere aplicar NMS.

YOLOv10 introduce una estrategia llamada $\textbf{Dual Label Assignment}$, que combina dos mecanismos:

- $\textbf{One-to-Many:}$ proporciona una señal de entrenamiento rica y robusta.
- $\textbf{One-to-One:}$ obliga al modelo a producir una sola predicción final por objeto.

#### Funcionamiento

Durante el entrenamiento, ambas ramas se optimizan simultáneamente. La rama O2M ayuda a aprender mejores representaciones, mientras que la rama O2O aprende a asociar cada objeto con una única predicción.

Durante la inferencia, YOLOv10 utiliza únicamente la rama $\textbf{one-to-one}$, por lo que:

- cada instancia real genera una sola predicción final,
- se elimina la necesidad de aplicar NMS,
- se evita que cajas de personas cercanas compitan entre sí mediante IoU.

#### Ventaja en escenas con oclusión

En el caso de los 15 clones superpuestos, YOLOv10 puede mantener detecciones separadas para cada persona porque el modelo ya fue entrenado para producir correspondencias directas entre objetos e inferencias finales. Así, no depende de una regla geométrica de supresión posterior que pueda eliminar individuos válidos por estar demasiado cerca unos de otros. YOLOv10 resuelve mejor el problema de oclusión porque aprende una correspondencia $\textit{una instancia \(\rightarrow\) una predicción}$, evitando el uso de NMS y reduciendo los falsos negativos en escenas densas.




# Task 3 - Prototipo Funcional: Pokedex en Tiempo Real

In [2]:
import cv2
import time
import numpy as np
from ultralytics import YOLO

# ── Model ──────────────────────────────────────────────────────────────────────
# YOLOv8n: nano variant pre-trained on COCO (80 common object classes).
# "nano" is the fastest variant; swap to "yolov8s.pt" or "yolov8m.pt" for more
# accuracy at the cost of speed.  No custom training required.
model = YOLO("yolov8n.pt")

# ── Hyperparameters ────────────────────────────────────────────────────────────
CONF_THRESHOLD = 0.40   # Confidence threshold – discard detections below this score
IOU_THRESHOLD  = 0.45   # NMS IoU threshold   – suppress overlapping boxes above this

# ── Video source ───────────────────────────────────────────────────────────────
# Use 0 for the default webcam.
# To use a pre-recorded file instead, replace 0 with the path, e.g.:
#   SOURCE = "mi_video_geek.mp4"
SOURCE = 0

cap = cv2.VideoCapture(SOURCE)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video source: {SOURCE}")

# ── Output writer ──────────────────────────────────────────────────────────────
fps_src = cap.get(cv2.CAP_PROP_FPS) or 30.0
w       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out_path = "task3_output.mp4"
fourcc   = cv2.VideoWriter_fourcc(*"mp4v")
writer   = cv2.VideoWriter(out_path, fourcc, fps_src, (w, h))

MAX_SECONDS   = 5
frame_limit   = int(fps_src * MAX_SECONDS)
frames_written = 0

prev_time = time.time()

while frames_written < frame_limit:
    ret, frame = cap.read()
    if not ret:
        break

    # ── FPS calculation (algorithmic, using time library) ──────────────────────
    curr_time = time.time()
    elapsed   = curr_time - prev_time
    fps       = 1.0 / elapsed if elapsed > 0 else 0.0
    prev_time = curr_time

    # ── Continuous inference ───────────────────────────────────────────────────
    # conf and iou are passed explicitly as required by the task.
    # verbose=False suppresses per-frame console output.
    results = model(frame, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD, verbose=False)

    # ── Tensor extraction ──────────────────────────────────────────────────────
    # results[0].boxes.xyxy  → (N, 4) tensor with (x1, y1, x2, y2) pixel coords
    # results[0].boxes.cls   → (N,)   tensor with integer class indices
    # results[0].boxes.conf  → (N,)   tensor with confidence scores
    boxes = results[0].boxes.xyxy.cpu().numpy()   # shape: (N, 4)
    cls   = results[0].boxes.cls.cpu().numpy()    # shape: (N,)
    confs = results[0].boxes.conf.cpu().numpy()   # shape: (N,)
    names = model.names                            # dict: {int → class_name}

    for (x1, y1, x2, y2), cls_id, conf in zip(boxes, cls, confs):
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        label = f"{names[int(cls_id)]} {conf:.2f}"

        # Draw bounding box using primitive OpenCV functions (NOT .plot())
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Label background for readability
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
        cv2.rectangle(frame, (x1, y1 - th - 6), (x1 + tw, y1), (0, 255, 0), cv2.FILLED)
        cv2.putText(frame, label, (x1, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)

    # ── FPS overlay (top-left corner) ─────────────────────────────────────────
    cv2.putText(frame, f"FPS: {fps:.1f}", (10, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 200, 255), 2, cv2.LINE_AA)

    writer.write(frame)
    frames_written += 1

cap.release()
writer.release()
print(f"Done — {frames_written} frames saved to '{out_path}'")

Done — 150 frames saved to 'task3_output.mp4'
